# College AI Chatbot - Multi PDF Chain

In [ ]:
!pip install -q langchain langchain-community chromadb pandas pypdf sentence-transformers langchain-groq langchain-huggingface python-dotenv

In [ ]:
import os, re, json, glob
import pandas as pd
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [ ]:
# load api keys
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("api key loaded" if GROQ_API_KEY else "ERROR: GROQ_API_KEY nahi mila .env me")

In [ ]:
# groq llm setup
llm = ChatGroq(
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    model_name="llama3-70b-8192"
)

# embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

NO_DATA = "Data available nahi hai."
print("llm and embeddings ready")

In [ ]:
# -------- WHATSAPP AGENT --------

# parse whatsapp chat file
def parse_chat(path):
    msgs = []
    pattern = r"(\d{1,2}/\d{1,2}/\d{2,4}),\s(\d{1,2}:\d{2}\s?[APMapm]{2})\s-\s([^:]+):\s(.*)"
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            m = re.match(pattern, line)
            if m:
                date, time, sender, msg = m.groups()
                if "<Media omitted>" not in msg:
                    msgs.append(f"[{date} {time}] {sender.strip()}: {msg.strip()}")
    return msgs

print("loading whatsapp chat...")
raw_msgs = parse_chat("chat_data.txt")

# chunk messages
chat_docs = [Document(page_content=m) for m in raw_msgs]
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chat_chunks = splitter.split_documents(chat_docs)

# store in chroma
chat_db = Chroma.from_documents(
    chat_chunks, embeddings,
    persist_directory="./chat_db_chain",
    collection_name="chat_chain_collection"
)
chat_retriever = chat_db.as_retriever(search_kwargs={"k": 5})

def whatsapp_agent(q):
    docs = chat_retriever.invoke(q)
    if not docs:
        return NO_DATA
    
    context = "\n".join([d.page_content for d in docs])
    prompt = f"""Tu ek WhatsApp group chat analyzer hai.
Neeche diye gaye chat data se question ka jawab de.
Answer Hindi/Hinglish me do.
Agar chat data me jawab na mile toh exactly bol: "Data available nahi hai."

CHAT DATA:
{context}

Question: {q}
Answer:"""
    return llm.invoke(prompt).content

print(f"whatsapp agent ready - {len(chat_chunks)} chunks, {len(raw_msgs)} messages")

In [ ]:
# -------- STUDENT AGENT --------

# load csv
df = pd.read_csv("akgec_data.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(".", "", regex=False)
print("columns:", list(df.columns))
print(f"total students: {len(df)}")

# prompt for llm to parse question into json
student_prompt = """You are an expert data analyst.

You are given:
1. A CSV dataset of students
2. A natural language question

Your task:
- Decide which column to search (lookup_column)
- Decide what value to search (lookup_value)
- Decide which column to return (target_column)

Return ONLY valid JSON. No explanation.

Available columns:
university_roll_no, student_name, branch, contact_no, email_address,
personal_email_id, course, 10th_%, 12th_%, btech_%

Example:
Question: "Shobhit ka phone number"
Return:
{"lookup_column": "student_name", "lookup_value": "Shobhit", "target_column": "contact_no"}

Example:
Question: "2300271540023 ka marks"
Return:
{"lookup_column": "university_roll_no", "lookup_value": "2300271540023", "target_column": "btech_%"}
"""

def parse_question(q):
    resp = llm.invoke(student_prompt + f"\nQuestion: {q}")
    text = resp.content.strip()
    # handle markdown code blocks
    if "```" in text:
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    return json.loads(text)

def search_csv(parsed):
    col = parsed["lookup_column"]
    val = str(parsed["lookup_value"])
    target = parsed["target_column"]
    
    result = df[df[col].astype(str).str.contains(val, case=False, na=False)]
    if result.empty:
        return NO_DATA
    
    row = result.iloc[0]
    name = row.get("student_name", val)
    return f"{name} ka {target.replace('_', ' ')} hai: {row[target]}"

def student_agent(q):
    try:
        parsed = parse_question(q)
        return search_csv(parsed)
    except:
        return NO_DATA

print("student agent ready")

In [ ]:
# -------- MULTI PDF AGENT --------

def clean(text):
    text = text.replace("\x0c", " ").replace("\n", " ")
    return re.sub(r"\s{2,}", " ", text).strip()

# load all pdfs from folder
pdf_folder = "pdf_file"
pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))
print(f"found {len(pdf_files)} pdfs")

all_docs = []
for path in pdf_files:
    name = os.path.basename(path)
    try:
        pages = PyPDFLoader(path).load()
        for i, p in enumerate(pages):
            all_docs.append(Document(
                page_content=clean(p.page_content),
                metadata={"page": p.metadata.get("page", i), "source": name}
            ))
        print(f"  loaded {name} - {len(pages)} pages")
    except Exception as e:
        print(f"  error loading {name}: {e}")

print(f"total pages: {len(all_docs)}")

# chunk and store
pdf_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
pdf_chunks = pdf_splitter.split_documents(all_docs)
print(f"total chunks: {len(pdf_chunks)}")

pdf_db = Chroma.from_documents(
    pdf_chunks, embeddings,
    persist_directory="./pdf_db_multipdf_chain",
    collection_name="multipdf_chain_collection"
)
pdf_retriever = pdf_db.as_retriever(search_kwargs={"k": 5})

# keyword search with token matching
stop_words = {"who", "is", "what", "the", "a", "an", "and", "in", "of",
              "tell", "me", "about", "his", "her", "their", "kaun",
              "hai", "kya", "ka", "ki", "ke", "describe", "how", "does",
              "do", "this", "that"}

def get_tokens(q):
    tokens = re.findall(r"[a-zA-Z0-9\.]+", q.lower())
    important = [t for t in tokens if t not in stop_words and len(t) > 1]
    return important if important else tokens

def clean_query(q):
    q = q.lower()
    for phrase in ["who is", "tell me about", "kaun hai", "kaun h", "what is", "describe"]:
        q = q.replace(phrase, "")
    return q.strip()

def keyword_search(q):
    # try exact substring first
    cq = clean_query(q)
    for d in all_docs:
        if cq in d.page_content.lower():
            return d.page_content
    
    # try matching all tokens
    tokens = get_tokens(q)
    best = None
    best_count = 0
    for d in all_docs:
        txt = d.page_content.lower()
        count = sum(1 for t in tokens if t in txt)
        if count == len(tokens) and count > best_count:
            best_count = count
            best = d.page_content
    if best:
        return best
    
    # partial match - at least half the tokens
    best = None
    best_count = 0
    min_match = max(1, len(tokens) // 2)
    for d in all_docs:
        txt = d.page_content.lower()
        count = sum(1 for t in tokens if t in txt)
        if count >= min_match and count > best_count:
            best_count = count
            best = d.page_content
    return best

def search_pdf(q):
    # keyword search first
    hit = keyword_search(q)
    if hit:
        return hit
    
    # vector search fallback
    docs = pdf_retriever.invoke(q)
    if docs:
        return "\n".join([d.page_content for d in docs[:3]])
    return None

def pdf_agent(q):
    context = search_pdf(q)
    if not context:
        return NO_DATA
    
    prompt = f"""Sirf neeche diye gaye data ka use karo.
Koi bhi extra baat mat jodo.
Answer Hindi/Hinglish me do.
Agar data me jawab na mile toh exactly bol: "Data available nahi hai."

DATA:
{context}

Question: {q}
Answer:"""
    return llm.invoke(prompt).content

print(f"pdf agent ready - {len(pdf_chunks)} chunks from {len(pdf_files)} pdfs")

In [ ]:
# -------- CHAIN LOGIC --------
# order: whatsapp -> student -> pdf

def college_chatbot_multipdf_chain(q):
    # try whatsapp first
    ans = whatsapp_agent(q)
    if NO_DATA.lower() not in ans.lower():
        print("\n[Chain -> WHATSAPP_AGENT]")
        return ans
    
    # then student csv
    ans = student_agent(q)
    if NO_DATA.lower() not in ans.lower():
        print("\n[Chain -> STUDENT_AGENT]")
        return ans
    
    # finally pdf
    ans = pdf_agent(q)
    if NO_DATA.lower() not in ans.lower():
        print("\n[Chain -> PDF_AGENT]")
        return ans
    
    print("\n[Chain -> NO MATCH]")
    return NO_DATA

print("chatbot ready!")

In [ ]:
# testing

print("="*50)
print("Q: library timing")
print(college_chatbot_multipdf_chain("library timing"))

print("\n" + "="*50)
print("Q: Shobhit ka number")
print(college_chatbot_multipdf_chain("Shobhit ka number"))

print("\n" + "="*50)
print("Q: hostel fees")
print(college_chatbot_multipdf_chain("hostel fees"))

print("\n" + "="*50)
print("Q: who talked most")
print(college_chatbot_multipdf_chain("who talked most"))

print("\n" + "="*50)
print("Q: who is S.L. Kapoor")
print(college_chatbot_multipdf_chain("who is S.L. Kapoor and what is his role in akgec"))

print("\n" + "="*50)
print("Q: placement statistics")
print(college_chatbot_multipdf_chain("placement statistics"))